# BigHaat Scraper (Fertilizer + Seeds)

This notebook scrapes product data from the given BigHaat collection links and exports a CSV.

Fields extracted:
- name
- image_link
- price
- rating
- description
- product_url
- category
- source_collection_url

In [ ]:
# If needed, run this once to install dependencies in the notebook kernel
%pip install -q requests pandas

In [ ]:
import json
import re
from urllib.parse import urlencode, urlsplit, urlunsplit, parse_qsl

import pandas as pd
import requests

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'
}

COLLECTIONS = {
    'fertilizer': [
        'https://www.bighaat.com/collections/plant-nutrition',
        'https://www.bighaat.com/collections/plant-nutrition',
        'https://www.bighaat.com/collections/organic-farming',
    ],
    'seeds': [
        'https://www.bighaat.com/collections/seeds-online',
    ],
}

In [ ]:
def add_or_replace_query_param(url: str, key: str, value: str) -> str:
    parts = list(urlsplit(url))
    query = dict(parse_qsl(parts[3], keep_blank_values=True))
    query[key] = value
    parts[3] = urlencode(query)
    return urlunsplit(parts)


def decode_next_f_chunks(html: str) -> str:
    chunks = []
    pattern = r'self\.__next_f\.push\(\[1,"(.*?)"\]\)</script>'

    for m in re.finditer(pattern, html, flags=re.DOTALL):
        raw = m.group(1)
        try:
            decoded = bytes(raw, 'utf-8').decode('unicode_escape')
        except Exception:
            decoded = raw
        chunks.append(decoded)

    return '\n'.join(chunks)


def extract_json_object_at(text: str, start_idx: int):
    if start_idx >= len(text) or text[start_idx] != '{':
        return None, start_idx

    depth = 0
    in_string = False
    escaped = False
    i = start_idx

    while i < len(text):
        ch = text[i]

        if in_string:
            if escaped:
                escaped = False
            elif ch == '\\':
                escaped = True
            elif ch == '"':
                in_string = False
        else:
            if ch == '"':
                in_string = True
            elif ch == '{':
                depth += 1
            elif ch == '}':
                depth -= 1
                if depth == 0:
                    raw_obj = text[start_idx:i+1]
                    return raw_obj, i + 1

        i += 1

    return None, start_idx


def extract_product_details_objects(decoded_text: str):
    needle = '"productDetails":'
    i = 0
    objects = []

    while True:
        idx = decoded_text.find(needle, i)
        if idx == -1:
            break

        obj_start = idx + len(needle)
        raw_obj, next_pos = extract_json_object_at(decoded_text, obj_start)

        if raw_obj:
            try:
                objects.append(json.loads(raw_obj))
            except json.JSONDecodeError:
                pass
            i = next_pos
        else:
            i = idx + len(needle)

    # Deduplicate by productId inside a page
    unique = {}
    for obj in objects:
        pid = obj.get('productId')
        if pid is not None and pid not in unique:
            unique[pid] = obj

    return list(unique.values())


def first_variant_price(product_obj):
    mv = product_obj.get('mapVariant') or {}
    if isinstance(mv, dict) and mv:
        first = next(iter(mv.values()))
        if isinstance(first, dict):
            return first.get('price')
    return None


def best_description(product_obj):
    desc = product_obj.get('productDescription')
    if isinstance(desc, str) and desc and not desc.startswith('$'):
        return desc

    for mf in product_obj.get('metafields', []) or []:
        if not isinstance(mf, dict):
            continue
        if mf.get('key') == 'description_tag':
            return mf.get('value')

    return None


def product_to_row(product_obj, category: str, source_collection_url: str):
    images = product_obj.get('productImages') or []
    image_link = images[0] if images else None

    rating_info = product_obj.get('ratingCumulativeInfo') or {}

    return {
        'product_id': product_obj.get('productId'),
        'name': product_obj.get('productName'),
        'image_link': image_link,
        'price': first_variant_price(product_obj),
        'rating': rating_info.get('average'),
        'description': best_description(product_obj),
        'product_url': product_obj.get('deepLinkUrl'),
        'category': category,
        'source_collection_url': source_collection_url,
    }


def scrape_collection(base_url: str, category: str, max_pages: int = 20):
    all_rows = []

    for page_no in range(1, max_pages + 1):
        page_url = add_or_replace_query_param(base_url, 'pageNo', str(page_no)) if page_no > 1 else base_url
        response = requests.get(page_url, headers=HEADERS, timeout=40)
        response.raise_for_status()

        decoded = decode_next_f_chunks(response.text)
        products = extract_product_details_objects(decoded)

        if not products:
            break

        page_rows = [product_to_row(p, category, base_url) for p in products]

        # Stop if this page repeats products seen in previous page(s).
        new_count = 0
        seen_ids = {r['product_id'] for r in all_rows}
        for row in page_rows:
            if row['product_id'] not in seen_ids:
                all_rows.append(row)
                seen_ids.add(row['product_id'])
                new_count += 1

        if new_count == 0:
            break

    return all_rows

In [ ]:
rows = []

for category, urls in COLLECTIONS.items():
    # Deduplicate repeated links provided in input
    unique_urls = list(dict.fromkeys(urls))

    for u in unique_urls:
        print(f'Scraping: {category} -> {u}')
        rows.extend(scrape_collection(u, category=category, max_pages=20))

df = pd.DataFrame(rows)
df = df.drop_duplicates(subset=['product_id', 'source_collection_url']).reset_index(drop=True)

print('Total records:', len(df))
df.head(10)

In [ ]:
# Final required columns
final_df = df[['name', 'image_link', 'price', 'rating', 'description', 'product_url', 'category', 'source_collection_url']]
final_df.to_csv('bighaat_products.csv', index=False, encoding='utf-8')

print('Saved:', 'bighaat_products.csv')
final_df.sample(min(10, len(final_df)))

: 